In [0]:
from pyspark.sql.functions import *
print("[STEP 0] ✅ Imports loaded successfully")

####Check the external data is accessible via Notebook

In [0]:
dbutils.fs.ls('/Volumes/vk_adb_caresync_workspace/landing/landing-volume/neon_postgres/hospitals/2026-10-07_08:05:25/')

####Reading the Parquet Data

In [0]:
df = spark.read.format('parquet').load('/Volumes/vk_adb_caresync_workspace/landing/landing-volume/neon_postgres/hospitals/2026-10-07_08:05:25/')

In [0]:
df.display()

#### Creating parameters

In [0]:
dbutils.widgets.text('source_system', ' ')
dbutils.widgets.text('source_table_name', ' ')
dbutils.widgets.text('landing_timestamp', ' ')
dbutils.widgets.text('source_file_format', ' ')

source_system = dbutils.widgets.get('source_system')
source_table_name = dbutils.widgets.get('source_table_name')
landing_timestamp = dbutils.widgets.get('landing_timestamp')
source_file_format = dbutils.widgets.get('source_file_format')

print("[STEP 1] ✅ Widget parameters loaded successfully")
print(f"  [PARAM] source_system      = '{source_system}'")
print(f"  [PARAM] source_table_name  = '{source_table_name}'")
print(f"  [PARAM] landing_timestamp  = '{landing_timestamp}'")
print(f"  [PARAM] source_file_format = '{source_file_format}'")

In [0]:
# Construct the landing path once and validate it exists
landing_path = f"/Volumes/vk_adb_caresync_workspace/landing/landing-volume/{source_system}/{source_table_name}/{landing_timestamp}/"

print("[STEP 2] Constructing and validating landing path")
print(f"  [PATH] {landing_path}")

try:
    files = dbutils.fs.ls(landing_path)
    print(f"  [RESULT] ✅ Path exists. Found {len(files)} file(s):")
    for f in files:
        print(f"    📄 {f.name} ({f.size} bytes)")
except Exception as e:
    print(f"  [WARNING] ❌ Landing path does NOT exist: {landing_path}")
    print(f"  [ERROR DETAIL] {str(e)}")
    # Check parent directories to give better diagnostics
    parent_path = f"/Volumes/vk_adb_caresync_workspace/landing/landing-volume/{source_system}/{source_table_name}/"
    print(f"  [INFO] Checking parent path: {parent_path}")
    try:
        parent_files = dbutils.fs.ls(parent_path)
        print(f"  [INFO] Parent path exists. Available timestamps:")
        for pf in parent_files:
            print(f"    📁 {pf.name}")
        print(f"  [INFO] But no data found for timestamp '{landing_timestamp}'")
    except:
        grandparent_path = f"/Volumes/vk_adb_caresync_workspace/landing/landing-volume/{source_system}/"
        print(f"  [INFO] Parent path also does not exist. Checking: {grandparent_path}")
        try:
            gp_files = dbutils.fs.ls(grandparent_path)
            print(f"  [INFO] Source system path exists. Available tables:")
            for gf in gp_files:
                print(f"    📁 {gf.name}")
            print(f"  [INFO] But table '{source_table_name}' was not found")
        except:
            print(f"  [INFO] Source system path also does not exist for '{source_system}'")
    print("\n  [EXIT] 🚪 No new files received. Exiting notebook gracefully.")
    dbutils.notebook.exit("No new files received")

In [0]:
if source_file_format == "csv":
    df_landing = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load(f"/Volumes/vk_adb_caresync_workspace/landing/landing-volume/{source_system}/{source_table_name}/{landing_timestamp}/")
else:
    df_landing = spark.read.format("parquet").load(f"/Volumes/vk_adb_caresync_workspace/landing/landing-volume/{source_system}/{source_table_name}/{landing_timestamp}/")

df_landing.display()

In [0]:
from pyspark.sql.functions import lit, current_timestamp

print("[STEP 4] Adding audit columns to landing DataFrame")
print(f"  [COLUMN] 'landing_timestamp' = lit('{landing_timestamp}')")
print(f"  [COLUMN] 'insert_timestamp'  = current_timestamp()")

df_landing_updated = df_landing.withColumn('landing_timestamp', lit(f'{landing_timestamp}')).withColumn('insert_timestamp', current_timestamp())

print(f"  [RESULT] ✅ Audit columns added")
print(f"  [SCHEMA] Updated columns ({len(df_landing_updated.columns)}): {df_landing_updated.columns}")
print("  [PREVIEW] Displaying updated DataFrame below:")
df_landing_updated.display()

In [0]:
target_table = f'vk_adb_caresync_workspace.bronze.{source_table_name}'
write_mode = 'append'

print("[STEP 5] Writing landing data to Bronze table")
print(f"  [TARGET TABLE] {target_table}")
print(f"  [WRITE MODE]   {write_mode}")
print(f"  [QUERY] df_landing_updated.write.mode('{write_mode}').saveAsTable('{target_table}')")

df_landing_updated.write.mode(write_mode).saveAsTable(target_table)

final_count = df_landing_updated.count()
print(f"  [RESULT] ✅ Write completed successfully")
print(f"  [RECORDS WRITTEN] {final_count} records appended to {target_table}")

In [0]:
%sql
SELECT * FROM vk_adb_caresync_workspace.bronze.hospitals;